## Review training job to identify GPU utilization bottlenecks
This notebook has a training example with a simple finetuning script using SFTTrainer. 
Considering the duration of workshop, we will use a trim version of `greenwich157/telco-5G-core-faults` dataset (180 records, split as train and test datasets).

Lab instuctor will provide a profiler report generated when this notebook was tested on `g6.12xlarge` multi-gpu instance that has `4 L4 NVIDIA GPUs`. The training job uses **model parallelism**. The goal of this lab is for you to review the existing report and code in this notebook, to identify the potential bottlenecks. Then identify best optimization technique that can be implemented on the instance provided to you. This notebook is shared with cell outputs printed, so you understand the configurations used while profiling. 
Feel free to make a copy of this notebook to implement and test your solution that can address the gpu utilization bottlenecks or you can use the notebook with sample solution provided below. 
### **Pls Note** : As the workshop Jupyterlab instance is running on a server with single accelerator/gpu, you cannot apply model parallelism to replicate the report similar to what the instructor shared. If this notebook is being used on your own AWS account, you may use g6.12xl or any other multi-gpu instance to run this notebook.

#### **Referrence Solution:** `lab-02/01-gpu-utilization/ft-tinyllama-gpu-utilization-solution.ipynb`

#### **IMPORTANT:** Some key observations from the report are shared as screenshots at the end of this notebook. 

#### **Pro-Tip:** use similar training parameters used in this notebook, to do a like to like comparison with the report the instructor shared with you (which is run on a multi-gpu instance)

In [ ]:
!pip install -U datasets transformers accelerate trl

In [ ]:
import torch
from datasets import Dataset, load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments, IntervalStrategy
from trl import SFTTrainer

dataset = "greenwich157/telco-5G-core-faults"
model_id= "TinyLlama/TinyLlama-1.1B-Chat-v1.0"

def prepare_training_data(dataset_id):
    data = load_dataset(dataset_id, split="train")
    data_df = data.to_pandas()
    data_df["text"] = data_df[["instruction", "input", "output"]].apply(lambda x:"<|system|>\n" + x.instruction + "</s><|user|>\n" + x.input + "</s>\n<|assistant|>\n" + x.output + "</s>", axis=1)
    data = Dataset.from_pandas(data_df)
    return data

full_train_data = prepare_training_data(dataset)
print ("training data prepared successfully ", full_train_data)
full_train_data = full_train_data.remove_columns(['instruction', 'input', 'output'])

full_train_data = full_train_data.select(range(180)) # trimming to reduce report size
full_train_data = full_train_data.train_test_split(test_size=0.20, seed=42)
print ("training data trimmed and split successfully ", full_train_data)
train_data = full_train_data['train']
test_data = full_train_data['test']

torch.cuda.empty_cache()

/rs-data/nv-ns-venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


training data prepared successfully  Dataset({
    features: ['instruction', 'input', 'output', 'text'],
    num_rows: 1323
})
training data trimmed and split successfully  DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 144
    })
    test: Dataset({
        features: ['text'],
        num_rows: 36
    })
})


In [ ]:
ft_model = AutoModelForCausalLM.from_pretrained(
    model_id, 
    device_map="auto",
    dtype=torch.float32
)

ft_train_args = TrainingArguments(
    batch_eval_metrics=True,
    disable_tqdm=False,
    dataloader_num_workers=4,
    dataloader_persistent_workers=True,
    dataloader_drop_last=True,
    dataloader_pin_memory=False,
    dataloader_prefetch_factor=2,
    do_eval=True,
    eval_accumulation_steps=4,
    eval_do_concat_batches=False,
    eval_strategy=IntervalStrategy.STEPS,
    full_determinism=True, 
    gradient_accumulation_steps=4,
    gradient_checkpointing=True,
    logging_strategy=IntervalStrategy.STEPS,
    logging_steps=6,
    learning_rate=2e-4,
    lr_scheduler_type="constant",
    max_grad_norm=1.0,
    num_train_epochs=2,
    optim="adamw_torch",
    output_dir="ft-tinyllama-fp32-multigpu-baseline",
    per_device_train_batch_size=16,
    per_device_eval_batch_size=4,
    prediction_loss_only=False,
    remove_unused_columns=False,
    skip_memory_metrics=False,
    save_strategy=IntervalStrategy.STEPS,
    save_steps=50,
    warmup_steps=0,
    warmup_ratio=0.1,
)
print ("training arguments initialized successfully ", ft_train_args)

ft_trainer = SFTTrainer(
    model=ft_model,
    args=ft_train_args,
    train_dataset=train_data, 
    eval_dataset=test_data,
)
print ("Trainer class initialized and staring full fine-tuning")

training arguments initialized successfully  TrainingArguments(
_n_gpu=4,
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adafactor=False,
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=True,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=True,
dataloader_num_workers=4,
dataloader_persistent_workers=True,
dataloader_pin_memory=False,
dataloader_prefetch_factor=2,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
eval_accumulation_steps=4,
eval_delay=0,
eval_do_concat_batches=False,
eval_on_start=False,
eval_steps=6,
eval_strategy=IntervalStrategy.S

Truncating eval dataset: 100%|██████████| 36/36 [00:00<00:00, 25531.78 examples/s]


Trainer class initialized and staring full fine-tuning


In [ ]:
torch.cuda.device_count()

4

In [ ]:
print(f"GPU 0 memory: {torch.cuda.get_device_properties(0).total_memory}")
print(f"GPU 0 allocated: {torch.cuda.memory_allocated(0)}")
print(f"GPU 1 memory: {torch.cuda.get_device_properties(0).total_memory}")
print(f"GPU 1 allocated: {torch.cuda.memory_allocated(0)}")
print(f"GPU 2 memory: {torch.cuda.get_device_properties(0).total_memory}")
print(f"GPU 2 allocated: {torch.cuda.memory_allocated(0)}")
print(f"GPU 3 memory: {torch.cuda.get_device_properties(0).total_memory}")
print(f"GPU 3 allocated: {torch.cuda.memory_allocated(0)}")

GPU 0 memory: 23670685696
GPU 0 allocated: 966852608
GPU 1 memory: 23670685696
GPU 1 allocated: 966852608
GPU 2 memory: 23670685696
GPU 2 allocated: 966852608
GPU 3 memory: 23670685696
GPU 3 allocated: 966852608


In [ ]:
print(ft_model.hf_device_map)

{'model.embed_tokens': 0, 'model.layers.0': 0, 'model.layers.1': 0, 'model.layers.2': 0, 'model.layers.3': 0, 'model.layers.4': 1, 'model.layers.5': 1, 'model.layers.6': 1, 'model.layers.7': 1, 'model.layers.8': 1, 'model.layers.9': 1, 'model.layers.10': 1, 'model.layers.11': 2, 'model.layers.12': 2, 'model.layers.13': 2, 'model.layers.14': 2, 'model.layers.15': 2, 'model.layers.16': 2, 'model.layers.17': 2, 'model.layers.18': 3, 'model.layers.19': 3, 'model.layers.20': 3, 'model.layers.21': 3, 'model.norm': 3, 'model.rotary_emb': 3, 'lm_head': 3}


In [ ]:
# Profile this cell only
ft_trainer.train()
print ("full fine-tuning completed successfully")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabli

Step,Training Loss,Validation Loss,Entropy,Num Tokens,Mean Token Accuracy
6,2.993700,2.789972,2.263372,33006.000000,0.485751


huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)
huggingface/tokenizers: The 

full fine-tuning completed successfully


Generating '/tmp/nsys-report-eb15.qdstrm'
[1/1] [========================100%] tmp_report.nsys-rep
Generated:
	/tmp/tmpatbklw8t/tmp_report.nsys-rep


## **Sample Profiling Report**

![](../../static-assets/lab-02/01-gpu-utilization-baseline-rep1.png)

![](../../static-assets/lab-02/01-gpu-utilization-baseline-rep2.png)

![](../../static-assets/lab-02/01-gpu-utilization-baseline-rep3.png)